# Qualidade de `feed_posts.csv`

Diagnóstico dos posts da rede social pelos mesmos cinco critérios usados nos outros arquivos, mais a checagem de privacidade do texto livre:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Respostas vazias ou sem conteúdo, e quem deveria estar no arquivo |
| **Validade** | Categorias, datas e números no domínio esperado? Texto livre que na prática é categoria converte? |
| **Consistência** | O texto concorda com as categorias e com os outros arquivos? |
| **Privacidade** | Há e-mail, telefone, CPF ou @ de terceiros no texto? |
| **Unicidade** | Cada post aparece uma vez só? |
| **Temporalidade** | As datas estão no período certo, antes do corte, no ritmo da base? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).
Transformar o texto em variáveis (extração com IA generativa, objetivo 5 do case) é outra etapa e não é feito aqui.

**Decisões já registradas que valem aqui** (`decisoes.md`): contas de teste e cópias de `usuarios.csv` removidas; datas
dos logs em horário de Brasília; nenhuma informação posterior ao corte. Os treinos vêm da tabela limpa de atividades (notebook 10), em horário de Brasília.

**Privacidade:** texto livre aparece só como contagem e respostas mais frequentes, nunca ao lado do `user_id`.

In [1]:
ARQUIVO = "feed_posts.csv"

import hashlib
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = [ARQUIVO, "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    # status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

# respostas sem conteúdo (vazio disfarçado), pelo checklist da skill
SEM_CONTEUDO = {"nada", "ok", "x", ".", "-", "asdf", "kkkk", "sla", "nao sei", "não sei", "nan", "null", "none", "n/a"}

def sem_conteudo(s):
    t = s.str.strip().str.lower()
    so_simbolo = t.ne("") & ~t.str.contains(r"[a-z0-9à-ú]", regex=True)   # só emoji ou pontuação
    return t.isin(SEM_CONTEUDO) | so_simbolo

PII = {
    "e-mail": r"[\w.+-]+@[\w-]+\.[a-z]{2,}",
    "telefone": r"\(?\b\d{2}\)?\s?9?\d{4}-?\d{4}\b",
    "CPF": r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
    "@ de perfil": r"(?<![\w.])@\w{3,}",
}

def contar_pii(s):
    return {nome: int(s.str.contains(rx, regex=True, case=False).sum()) for nome, rx in PII.items()}

brutos_usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False, usecols=["user_id", "email"])
email = brutos_usuarios.set_index("user_id")["email"].str.strip()
TESTE = set(email.index[email.str.lower().str.endswith("@vitaliza.com.br")])
COPIA = set(email.index[email != email.str.lower()])
usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro", "canal_aquisicao"]).set_index("user_id")

def contas_invalidas(df, criterio="Completude"):
    n_teste, n_copia = int(df["user_id"].isin(TESTE).sum()), int(df["user_id"].isin(COPIA).sum())
    sem_cad = int((~df["user_id"].isin(email.index)).sum())
    achado(criterio, "linhas de contas de teste, cópias ou sem cadastro",
           f"teste: {n_teste}; cópias: {n_copia}; sem cadastro: {sem_cad}", n_teste + n_copia + sem_cad, "Se houver: excluir")

## 0. Leitura

In [2]:
feed = pd.read_csv(BRUTOS / ARQUIVO, dtype=str, keep_default_na=False)
assert len(feed) == 119_309  # cabeçalho da seção 12 do dicionário
N = len(feed)
print(f"feed: {N:,} linhas, {feed['user_id'].nunique():,} autores -> {list(feed.columns)}")
feed["dh"] = pd.to_datetime(feed["data_hora"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
feed["curtidas_n"] = pd.to_numeric(feed["curtidas"], errors="coerce")
feed["texto"].value_counts().head(10)

feed: 119,309 linhas, 26,899 autores -> ['post_id', 'user_id', 'data_hora', 'texto', 'curtidas']


texto
funcional em casa            4083
yoga pra fechar o dia        3982
aula de hoje pagou           3925
perna destruída              3886
treino de força feito        3389
treino rápido de 30 min      3328
mais um                      1903
pace bom hoje                1781
treino de tiro hoje          1749
corrida pra começar o dia    1522
Name: count, dtype: int64

## 1. Completude

Todo post tem texto. `[post removido]` é um marcador, não um texto: o conteúdo sumiu, mas a linha e as curtidas ficaram.

In [3]:
vazio = int((feed["texto"].str.strip() == "").sum())
lixo = int(sem_conteudo(feed["texto"]).sum())
achado("Completude", "texto vazio ou sem conteúdo", f"{vazio} vazios e {lixo} sem conteúdo", vazio + lixo,
       "Nenhum" if vazio + lixo == 0 else "Tratar como sem texto")
removido = feed["texto"].str.strip().eq("[post removido]")
print(f"[post removido]: {removido.sum():,}; curtidas medianas {feed.loc[removido, 'curtidas_n'].median()} "
      f"(demais: {feed.loc[~removido, 'curtidas_n'].median()}); por mês: "
      f"{feed.loc[removido, 'dh'].dt.to_period('M').value_counts().sort_index().astype(int).to_dict()}")
achado("Completude", "post com o texto removido ([post removido])",
       f"{removido.sum():,} posts, espalhados por todos os meses, com curtidas iguais às dos demais: o post existiu e o texto foi apagado depois",
       int(removido.sum()), "Contar o post (atividade na rede) e tratar o texto como ausente; não enviar o marcador para a extração")
contas_invalidas(feed)

[post removido]: 1,214; curtidas medianas 6.0 (demais: 6.0); por mês: {Period('2025-07', 'M'): 68, Period('2025-08', 'M'): 81, Period('2025-09', 'M'): 71, Period('2025-10', 'M'): 71, Period('2025-11', 'M'): 81, Period('2025-12', 'M'): 91, Period('2026-01', 'M'): 112, Period('2026-02', 'M'): 64, Period('2026-03', 'M'): 85, Period('2026-04', 'M'): 78, Period('2026-05', 'M'): 108, Period('2026-06', 'M'): 73, Period('2026-07', 'M'): 68, Period('2026-08', 'M'): 79, Period('2026-09', 'M'): 83, Period('2026-10', 'M'): 1}


## 2. Validade

In [4]:
falha = int(feed["dh"].isna().sum())
achado("Validade", "data_hora que não converte", f"{falha}", falha, "Nenhum")
curt = feed["curtidas_n"]
print("curtidas:", curt.describe().round(1).to_dict())
invalida = int((curt.isna() | (curt < 0) | (curt % 1 != 0)).sum())
achado("Validade", "curtidas não inteiras ou negativas", f"{invalida}; de {int(curt.min())} a {int(curt.max())}", invalida, "Nenhum")

curtidas: {'count': 119309.0, 'mean': 7.2, 'std': 4.5, 'min': 0.0, '25%': 4.0, '50%': 6.0, '75%': 8.0, 'max': 38.0}


## 3. Consistência

### Fuso de `data_hora`

O dicionário não diz o fuso. Os posts saem "quase sempre logo depois de um treino". Teste: para cada post de 2026 (período
coberto pelos treinos), o último treino do autor antes do post. Se o post estiver em horário de Brasília, a distância é curta
e positiva; se estiver em UTC sem o Z, aparece um deslocamento de 3h.

In [5]:
ativ = pd.read_parquet(PROCESSADOS / "atividades_limpa.parquet", columns=["user_id", "inicio", "duracao_min", "origem"])
ativ = ativ.sort_values("inicio")
posts26 = feed[(feed["dh"] >= "2026-01-01") & (feed["dh"] < FIM_CORTE)].sort_values("dh")

def distancia(deslocamento_h):
    p = posts26.assign(t=posts26["dh"] - pd.Timedelta(hours=deslocamento_h)).sort_values("t")
    m = pd.merge_asof(p, ativ, left_on="t", right_on="inicio", by="user_id", direction="backward",
                      tolerance=pd.Timedelta(hours=24))
    horas = (m["t"] - m["inicio"]).dt.total_seconds() / 3600
    return horas

hipoteses = {}
for nome, desloc in [("como está (Brasília)", 0), ("se fosse UTC (-3h)", 3)]:
    h = distancia(desloc)
    hipoteses[nome] = {"posts com treino nas 24h antes": int(h.notna().sum()),
                       "até 3h depois do início do treino": int(h.between(0, 3).sum()),
                       "mediana (h)": round(float(h.median()), 1)}
display(pd.DataFrame(hipoteses).T)
hora_post = feed["dh"].dt.hour
hora_treino = ativ.loc[ativ["origem"] != "garmin", "inicio"].dt.hour if "origem" in ativ else ativ["inicio"].dt.hour
print(f"hora dos posts: {hora_post.min()}h a {hora_post.max()}h; hora dos treinos (Brasília): {hora_treino.min()}h a {hora_treino.max()}h")
br, utc = hipoteses["como está (Brasília)"], hipoteses["se fosse UTC (-3h)"]
achado("Consistência", "fuso de data_hora (o dicionário não diz)",
       f"como está: {br['até 3h depois do início do treino']:,} posts até 3h depois do início de um treino; com -3h: "
       f"{utc['até 3h depois do início do treino']:,}; posts das {hora_post.min()}h às {hora_post.max()}h, mesma faixa dos treinos "
       "em Brasília (em UTC, seriam das 3h às 19h)", 0,
       "Usar data_hora como horário de Brasília" if br["até 3h depois do início do treino"] >= utc["até 3h depois do início do treino"]
       else "Converter de UTC para Brasília", status="ok")

,posts com treino nas 24h antes,até 3h depois do início do treino,mediana (h)
como está (Brasília),43694.0,10629.0,7.7
se fosse UTC (-3h),37636.0,8307.0,7.9


hora dos posts: 6h a 22h; hora dos treinos (Brasília): 4h a 22h


In [6]:
antes = int((feed["dh"].dt.normalize() < feed["user_id"].map(usuarios["data_cadastro"])).sum())
achado("Consistência", "post antes do dia do cadastro do autor", f"{antes}", antes, "Nenhum")

## 4. Privacidade

In [7]:
pii = contar_pii(feed["texto"])
print(pii)
achado("Privacidade", "e-mail, telefone, CPF ou @ no texto", f"{pii}", sum(pii.values()), "Se houver: mascarar antes de qualquer envio a IA")

{'e-mail': 0, 'telefone': 0, 'CPF': 0, '@ de perfil': 0}


## 5. Unicidade

O mesmo texto curto ("funcional em casa") é natural entre pessoas e em dias diferentes. Duplicata seria o mesmo autor, com o
mesmo texto, no mesmo minuto.

In [8]:
achado("Unicidade", "post_id repetido", f"{int(feed['post_id'].duplicated().sum())}", int(feed["post_id"].duplicated().sum()), "Nenhum")
mesmo = feed.duplicated(["user_id", "texto", "dh"], keep="first")
mesmo_dia = feed.assign(d=feed["dh"].dt.normalize()).duplicated(["user_id", "texto", "d"], keep="first") & ~removido
print(f"mesmo autor, texto e minuto: {mesmo.sum()}; mesmo autor e texto no mesmo dia (sem os removidos): {mesmo_dia.sum()}")
achado("Unicidade", "mesmo autor, texto e minuto", f"{int(mesmo.sum())}", int(mesmo.sum()), "Se houver: manter um")
achado("Unicidade", "mesmo autor e texto no mesmo dia",
       f"{int(mesmo_dia.sum())} posts (textos curtos e comuns, horários diferentes)", int(mesmo_dia.sum()),
       "Nenhum: dois treinos no mesmo dia rendem dois posts parecidos", status="esperado" if mesmo_dia.sum() else "ok")

mesmo autor, texto e minuto: 0; mesmo autor e texto no mesmo dia (sem os removidos): 0


## 6. Temporalidade

Posts de 01/07/2025 até o corte (dicionário). As curtidas não têm data: são as acumuladas até a exportação.

In [9]:
print(f"data_hora de {feed['dh'].min()} a {feed['dh'].max()}")
pos = feed["dh"] >= FIM_CORTE
print("depois do corte, por dia:", feed.loc[pos, "dh"].dt.strftime("%d/%m").value_counts().sort_index().to_dict())
achado("Temporalidade", "post depois do corte", f"{int(pos.sum())} posts de 01/10 a {feed.loc[pos, 'dh'].max():%d/%m/%Y}",
       int(pos.sum()), "Excluir posts depois de 30/09/2026")
antes_ini = int((feed["dh"] < pd.Timestamp("2025-07-01")).sum())
achado("Temporalidade", "post antes de 01/07/2025", f"{antes_ini}", antes_ini, "Nenhum")
mensal = feed.loc[~pos].groupby(feed["dh"].dt.to_period("M")).size()
var = (100 * (mensal / mensal.shift() - 1)).round(0)
print("posts por mês:", mensal.astype(int).to_dict())
achado("Temporalidade", "mês sem post ou com salto acima de 50%", f"{int((var.abs() > 50).sum())}", int((var.abs() > 50).sum()), "Nenhum")
curtidas_recentes = feed.loc[~pos].groupby(feed["dh"].dt.to_period("M"))["curtidas_n"].mean().round(1)
print("curtidas médias por mês do post:", curtidas_recentes.to_dict())
achado("Temporalidade", "curtidas sem data (acumuladas até a exportação)",
       f"as curtidas de cada post são as da exportação, sem data; média de {curtidas_recentes.min()} a {curtidas_recentes.max()} por mês",
       N, "Não usar curtidas como preditor no corte sem considerar que podem incluir curtidas posteriores; preferir o número de posts")

data_hora de 2025-07-01 06:07:55 a 2026-10-07 18:52:16
depois do corte, por dia: {'01/10': 62, '02/10': 31, '03/10': 16, '04/10': 5, '05/10': 2, '06/10': 4, '07/10': 2}
posts por mês: {Period('2025-07', 'M'): 6804, Period('2025-08', 'M'): 7319, Period('2025-09', 'M'): 7356, Period('2025-10', 'M'): 7832, Period('2025-11', 'M'): 7664, Period('2025-12', 'M'): 8070, Period('2026-01', 'M'): 8832, Period('2026-02', 'M'): 7317, Period('2026-03', 'M'): 8347, Period('2026-04', 'M'): 8058, Period('2026-05', 'M'): 8733, Period('2026-06', 'M'): 7851, Period('2026-07', 'M'): 7893, Period('2026-08', 'M'): 8359, Period('2026-09', 'M'): 8752}


curtidas médias por mês do post: {Period('2025-07', 'M'): 7.2, Period('2025-08', 'M'): 7.1, Period('2025-09', 'M'): 7.0, Period('2025-10', 'M'): 7.1, Period('2025-11', 'M'): 7.1, Period('2025-12', 'M'): 8.2, Period('2026-01', 'M'): 7.1, Period('2026-02', 'M'): 7.1, Period('2026-03', 'M'): 7.1, Period('2026-04', 'M'): 7.1, Period('2026-05', 'M'): 7.2, Period('2026-06', 'M'): 7.2, Period('2026-07', 'M'): 7.2, Period('2026-08', 'M'): 7.2, Period('2026-09', 'M'): 7.0}


## Resumo

In [10]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / f"qualidade_{Path(ARQUIVO).stem}.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 12, 'atenção': 3}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,texto vazio ou sem conteúdo,ok,0 vazios e 0 sem conteúdo,0,Nenhum,0.00
1,Completude,post com o texto removido ([post removido]),atenção,"1,214 posts, espalhados por todos os meses, com curtidas iguais às...",1214,Contar o post (atividade na rede) e tratar o texto como ausente; n...,1.02
2,Completude,"linhas de contas de teste, cópias ou sem cadastro",ok,teste: 0; cópias: 0; sem cadastro: 0,0,Se houver: excluir,0.00
3,Validade,data_hora que não converte,ok,0,0,Nenhum,0.00
4,Validade,curtidas não inteiras ou negativas,ok,0; de 0 a 38,0,Nenhum,0.00
5,Consistência,fuso de data_hora (o dicionário não diz),ok,"como está: 10,629 posts até 3h depois do início de um treino; com ...",0,Usar data_hora como horário de Brasília,0.00
6,Consistência,post antes do dia do cadastro do autor,ok,0,0,Nenhum,0.00
7,Privacidade,"e-mail, telefone, CPF ou @ no texto",ok,"{'e-mail': 0, 'telefone': 0, 'CPF': 0, '@ de perfil': 0}",0,Se houver: mascarar antes de qualquer envio a IA,0.00
8,Unicidade,post_id repetido,ok,0,0,Nenhum,0.00
9,Unicidade,"mesmo autor, texto e minuto",ok,0,0,Se houver: manter um,0.00


In [11]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
